# Congklak AlphaZero Training (Kaggle Version)
### Metode: gdown (Initial Download) + WandB (Automatic Auto-Save)

**Persiapan WandB:**
1. Buka menu **Add-ons** -> **Secrets** di sidebar kanan.
2. Tambahkan Secret baru:
   - Label: `WANDB_API_KEY` 
   - Value: (Tempel API Key yang baru saja Anda buat)
3. Pastikan checkbox **Attached** sudah dicentang.

In [ ]:
# 1. Install dependencies
!pip install --upgrade gdown wandb

import os, sys, torch, wandb
from kaggle_secrets import UserSecretsClient

# 2. Setup WandB (Ambil API Key dari Kaggle Secrets)
user_secrets = UserSecretsClient()
try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("WandB logged in successfully!")
except:
    print("PERINGATAN: WANDB_API_KEY tidak ditemukan di Kaggle Secrets. Silahkan tambahkan di menu Add-ons.")

repo_url = "https://github.com/billdansr/AlphaDDA.git"
local_repo_path = "/kaggle/working/AlphaDDA"
game_subdir = "AlphaZero/Congklak"

# 3. Clone Repository
if not os.path.exists(local_repo_path):
    !git clone {repo_url} {local_repo_path}

local_game_path = os.path.join(local_repo_path, game_subdir)
os.chdir(local_game_path)

# 4. Download Model Terbaru dari GDrive (Opsional: Jika ingin melanjutkan training)
# Pastikan file di GDrive sudah di-set 'Anyone with the link can view'
GDRIVE_FILE_ID = '1F7ssgFZpEYJu8MhN3a0RNjk6d6o-EyBM' 

print("--- VERIFIKASI CHECKPOINT ---")
if os.path.exists('checkpoint.model'):
    print("✅ File 'checkpoint.model' ditemukan di direktori kerja.")
elif GDRIVE_FILE_ID != 'MASUKKAN_FILE_ID_DISINI':
    print(f"📡 Mencoba mengunduh model dari GDrive (ID: {GDRIVE_FILE_ID})...")
    try:
        !gdown --id {GDRIVE_FILE_ID} -O checkpoint.model
        if os.path.exists('checkpoint.model'):
            print("✅ Berhasil mengunduh checkpoint.")
        else:
            print("❌ Gagal mengunduh. Periksa apakah ID benar dan akses file sudah 'Anyone with the link'.")
    except Exception as e:
        print(f"❌ Error saat gdown: {e}")
else:
    print("⚠️ GDRIVE_FILE_ID belum diisi. Jika ini sengaja, training akan mulai dari Iterasi 0.")

# Pastikan permission file benar untuk eksekusi script
!chmod +x *.model 2>/dev/null || true

In [ ]:
# 5. Jalankan Training
%env PYTHONPATH=.:$PYTHONPATH

# Inisialisasi WandB untuk tracking eksperimen skripsi
run = wandb.init(
    project="AlphaZero-Congklak", 
    name=f"train-{os.getenv('KAGGLE_KERNEL_RUN_TYPE', 'interactive')}"
)

!python train_mp.py

# 6. Simpan Model ke WandB secara otomatis
print("\n--- UPLOADING MODELS TO WANDB ---")
wandb.save("*.model")
run.finish()
print("Selesai! Model aman di cloud WandB.")

In [ ]:
# 7. Evaluasi (Opsional)
%cd /kaggle/working/AlphaDDA/AlphaDDA1/Congklak
# Ambil model hasil training tadi
!cp /kaggle/working/AlphaDDA/AlphaZero/Congklak/checkpoint.model .
!python test_dda.py 1 300